In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf

In [ ]:
path_file = "/content/Obesity_Dataset.xlsx"

data = pd.read_excel(path_file)
data

In [ ]:
sns.countplot(x="Class", data=data)

In [4]:
corr = data.corr()
correlation_with_obesity = corr.loc['Class'].drop('Class').sort_values(ascending=False)
correlation_with_obesity

column_name = []
for i, value in enumerate(correlation_with_obesity):
  if value > 0:
    print(f"{correlation_with_obesity.index[i]}: {value}")
    column_name.append(correlation_with_obesity.index[i])

Age: 0.5798120169074363
Number_of_Main_Meals_Daily: 0.5093900118895416
Physical_Excercise: 0.3893545975077366
Height: 0.16156293603237643
Food_Intake_Between_Meals: 0.12932735134765314


In [ ]:
for name in column_name:
  plt.figure(figsize=(12, 6))
  sns.barplot(x="Class", y=name, data=data)
  plt.title(f'Relationship between {name.replace("_", " ")} and Obesity Class')

In [6]:
X = data.drop('Class', axis=1).values
y = data['Class'].values
y = y.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
scaler = MinMaxScaler(feature_range=(1, 5))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = y_train - 1
y_test = y_test - 1
num_classes = len(np.unique(y_train))

In [8]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('val_accuracy') is not None and logs.get('val_accuracy') > 0.8:
            print("\nReached 80.0% val_accuracy so cancelling training!")
            self.model.stop_training = True

In [9]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [10]:
hist = model.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_data=(X_test, y_test), callbacks=[myCallback()],
    verbose=0)

In [11]:
acc = model.evaluate(X_test, y_test)[1]
print("Test Accuracy: {:.2f}%".format(acc * 100))

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7890 - loss: 1.4346 
Test Accuracy: 76.71%
